# 02B - Trade-Finance / Contingent Facilities LGD

This notebook adds a dedicated trade-finance / contingent-facilities LGD segment under the parent commercial cash-flow framework.

Product examples:
- bank guarantees
- contingent obligations
- trade-related short-term facilities


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.commercial_data_controls import assign_framework_segment
from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(62)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy:** observed workout inputs -> approved proxies -> conservative fallback with disclosure.
- **Proxy logic:** contingent conversion, claim probability, and cash-security strength are synthetic but transparent.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** this is a portfolio-project proxy baseline, not calibrated to internal guarantee-claim history.


## Objective
Build an interview-ready trade-finance / contingent-facilities LGD segment focused on contingent conversion to EAD, security/cash support, and recovery timing.

## Drivers
- product type and transaction type
- contingent vs already drawn status
- claim/crystallisation probability
- tenor
- cash security and collateral support
- customer risk strength

## Logic
- EAD is driven by contingent conversion plus already drawn balances.
- LGD is driven by security/cash backing, customer strength, legal/processing costs, and recovery timing.

## Downturn
- Stress increases claim conversion and conversion-to-funded EAD, weakens security benefit, and lengthens recovery timing.

## Validation
- Bounds checks on contingent amounts, claim probability, EAD floor vs drawn, and recovery-time sanity.

## Limitations
- Synthetic proxy assumptions; production use requires internal claim and recovery evidence.


## Why This Segment Differs from Standard Term Loans

- Exposure may be **contingent** and crystallise into funded EAD when claims are called.
- Tenor is often shorter and transaction-linked.
- Some facilities are fully or partially **cash-secured**.
- Workout path can be faster/structurally different than longer-term amortising loans.


In [ ]:
# Build trade / contingent segment on the same base dataset as parent commercial notebook
all_loans, _ = generate_commercial_data(n_loans=420, seed=43)
all_loans = all_loans.copy()

all_loans['icr'] = pd.to_numeric(all_loans['interest_coverage_ratio'], errors='coerce')
all_loans['industry_risk_bucket'] = all_loans['industry_risk_level'].fillna('Medium')
all_loans['watchlist_flag'] = (
    all_loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (all_loans['icr'] < 1.35)
).astype(int)

# Parent framework mapping alignment (must match parent notebook)
all_loans['framework_segment'] = assign_framework_segment(all_loans)

trade = all_loans[all_loans['framework_segment'] == 'Trade / Contingent Facilities (Proxy)'].copy()

print('All loans:', len(all_loans))
print('Trade/contingent segment loans:', len(trade))
display(trade[['loan_id', 'facility_type', 'security_type', 'ead']].head(10))


In [ ]:
# Trade/contingent product and risk structure proxies
rng = np.random.default_rng(62)

trade['product_type'] = rng.choice(
    ['Bank Guarantee', 'Performance Bond', 'Standby LC', 'Trade Contingent Facility'],
    size=len(trade),
    p=[0.40, 0.25, 0.20, 0.15],
)

trade['transaction_type_proxy'] = rng.choice(
    ['Domestic Trade', 'Import / Export', 'Construction Contract', 'General Corporate Obligation'],
    size=len(trade),
    p=[0.35, 0.25, 0.25, 0.15],
)

trade['is_contingent_flag'] = np.where(trade['product_type'].isin(['Bank Guarantee', 'Performance Bond', 'Standby LC']), 1, 0)
trade['tenor_months'] = rng.integers(3, 25, size=len(trade))

industry_risk_factor = trade['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)

trade['cash_security_pct'] = (
    0.28
    + 0.42 * (trade['product_type'] == 'Bank Guarantee').astype(int)
    + 0.12 * (trade['product_type'] == 'Standby LC').astype(int)
    - 0.15 * trade['watchlist_flag']
    + rng.normal(0.0, 0.10, len(trade))
).clip(0.0, 1.0)

trade['collateral_support_pct'] = (
    0.20
    + 0.50 * trade['security_coverage_ratio'].clip(0.0, 1.4)
    + 0.08 * (1 - industry_risk_factor)
    + rng.normal(0.0, 0.08, len(trade))
).clip(0.0, 1.0)

trade['security_level'] = np.select(
    [trade['cash_security_pct'] >= 0.95, trade['cash_security_pct'] >= 0.40],
    ['Fully Cash-Secured', 'Partially Secured'],
    default='Unsecured / Weakly Secured',
)

trade['customer_risk_strength'] = (
    0.60
    + 0.18 * ((trade['icr'] - 1.20) / 2.00).clip(-0.6, 1.0)
    - 0.20 * trade['watchlist_flag']
    - 0.12 * industry_risk_factor
    + rng.normal(0.0, 0.08, len(trade))
).clip(0.05, 1.00)

# Contingent and drawn profile
trade['facility_limit_proxy'] = trade['facility_limit'].clip(lower=trade['drawn_balance'])

trade['already_drawn_amount'] = (
    trade['drawn_balance']
    * np.where(trade['is_contingent_flag'] == 1, rng.uniform(0.10, 0.55, len(trade)), rng.uniform(0.60, 1.00, len(trade)))
).clip(lower=0.0)
trade['already_drawn_amount'] = np.minimum(trade['already_drawn_amount'], trade['facility_limit_proxy'])

trade['undrawn_headroom_amount'] = (trade['facility_limit_proxy'] - trade['already_drawn_amount']).clip(lower=0.0)

trade['contingent_commitment_amount'] = (
    trade['undrawn_headroom_amount']
    * np.where(trade['is_contingent_flag'] == 1, rng.uniform(0.75, 1.00, len(trade)), rng.uniform(0.20, 0.70, len(trade)))
).clip(lower=0.0)

trade['claim_probability_base'] = (
    0.08
    + 0.18 * trade['watchlist_flag']
    + 0.12 * (1.0 - trade['customer_risk_strength'])
    + 0.10 * industry_risk_factor
    + 0.06 * ((trade['tenor_months'] - 6) / 18.0).clip(0.0, 1.0)
    + 0.05 * (trade['is_contingent_flag'])
    + rng.normal(0.0, 0.03, len(trade))
).clip(0.01, 0.95)

display(
    trade[
        [
            'product_type', 'is_contingent_flag', 'tenor_months',
            'cash_security_pct', 'collateral_support_pct', 'security_level',
            'customer_risk_strength', 'facility_limit_proxy',
            'already_drawn_amount', 'undrawn_headroom_amount', 'contingent_commitment_amount',
            'claim_probability_base',
        ]
    ].head(12)
)


In [ ]:
# EAD conversion logic (base and downturn)
trade['conversion_factor_base'] = (
    trade['claim_probability_base']
    * (1.0 + 0.05 * trade['watchlist_flag'] + 0.04 * ((trade['tenor_months'] - 6) / 18.0).clip(0.0, 1.0))
).clip(0.01, 1.00)

trade['ead_from_contingent_base'] = trade['contingent_commitment_amount'] * trade['conversion_factor_base']
trade['ead_base'] = trade['already_drawn_amount'] + trade['ead_from_contingent_base']
trade['ead_base'] = np.maximum(trade['ead_base'], trade['already_drawn_amount'])
trade['ead_base'] = np.minimum(trade['ead_base'], trade['facility_limit_proxy'])

trade['claim_probability_downturn'] = (
    trade['claim_probability_base']
    + 0.08
    + 0.10 * trade['watchlist_flag']
    + 0.06 * (1.0 - trade['customer_risk_strength'])
).clip(0.02, 1.00)

trade['conversion_factor_downturn'] = (
    trade['claim_probability_downturn']
    * (1.08 + 0.08 * trade['watchlist_flag'])
).clip(0.02, 1.00)

trade['ead_from_contingent_downturn'] = trade['contingent_commitment_amount'] * trade['conversion_factor_downturn']
trade['ead_downturn'] = trade['already_drawn_amount'] * (1.0 + 0.03 * trade['watchlist_flag']) + trade['ead_from_contingent_downturn']
trade['ead_downturn'] = np.maximum(trade['ead_downturn'], trade['already_drawn_amount'])
trade['ead_downturn'] = np.minimum(trade['ead_downturn'], trade['facility_limit_proxy'])

trade['ead_conversion_uplift_pct'] = (
    (trade['ead_base'] - trade['already_drawn_amount']) / trade['already_drawn_amount'].replace(0, np.nan)
).fillna(0.0).clip(0.0, 5.0)

ead_conversion_summary = (
    trade.groupby('security_level', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_contingent_commitment': g['contingent_commitment_amount'].sum(),
                'total_already_drawn': g['already_drawn_amount'].sum(),
                'total_ead_base': g['ead_base'].sum(),
                'total_ead_downturn': g['ead_downturn'].sum(),
                'ead_weighted_claim_prob_base': exposure_weighted_average(g, 'claim_probability_base', 'ead_base'),
                'ead_weighted_claim_prob_downturn': exposure_weighted_average(g, 'claim_probability_downturn', 'ead_base'),
                'ead_weighted_conversion_uplift_pct': exposure_weighted_average(g, 'ead_conversion_uplift_pct', 'ead_base'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('total_ead_base', ascending=False)
)

print('=== EAD Conversion Summary ===')
display(ead_conversion_summary.round(4))


In [ ]:
# LGD logic (base/downturn/final)
security_level_factor = trade['security_level'].map(
    {'Fully Cash-Secured': 0.0, 'Partially Secured': 0.6, 'Unsecured / Weakly Secured': 1.2}
).fillna(0.6)

trade['post_claim_recovery_months'] = (
    4
    + 10 * (1.0 - trade['cash_security_pct'])
    + 6 * trade['watchlist_flag']
    + 3 * security_level_factor
    + np.random.default_rng(620).normal(0.0, 1.8, len(trade))
).clip(2.0, 36.0)

trade['legal_processing_cost_pct'] = (
    0.010
    + 0.040 * (1.0 - trade['cash_security_pct'])
    + 0.020 * trade['watchlist_flag']
    + 0.010 * security_level_factor
).clip(0.005, 0.18)

trade['lgd_base_economic'] = (
    0.38 * trade['realised_lgd']
    + 0.27 * (1.0 - trade['cash_security_pct'])
    + 0.14 * (1.0 - trade['collateral_support_pct'])
    + 0.11 * (1.0 - trade['customer_risk_strength'])
    + 0.04 * trade['watchlist_flag']
    + 0.04 * ((trade['post_claim_recovery_months'] - 6.0) / 18.0).clip(0.0, 1.0)
    + trade['legal_processing_cost_pct']
).clip(0.05, 0.92)

trade['post_claim_recovery_months_downturn'] = (
    trade['post_claim_recovery_months']
    * (1.12 + 0.10 * trade['watchlist_flag'] + 0.06 * security_level_factor)
).clip(2.0, 48.0)

trade['downturn_scalar'] = (
    1.00
    + 0.08 * trade['claim_probability_downturn']
    + 0.10 * (1.0 - trade['cash_security_pct'])
    + 0.05 * ((trade['post_claim_recovery_months_downturn'] - trade['post_claim_recovery_months']) / 24.0).clip(0.0, 1.0)
).clip(1.00, 1.45)

trade['lgd_downturn'] = (trade['lgd_base_economic'] * trade['downturn_scalar']).clip(0.0, 1.0)
trade['moc_addon'] = 0.015 + 0.010 * trade['watchlist_flag']
trade['supervisory_floor_proxy'] = trade['security_level'].map(
    {'Fully Cash-Secured': 0.03, 'Partially Secured': 0.18, 'Unsecured / Weakly Secured': 0.35}
).fillna(0.25)
trade['lgd_final_regulatory'] = np.maximum(trade['lgd_downturn'] + trade['moc_addon'], trade['supervisory_floor_proxy']).clip(0.0, 1.0)


def weighted_lgd(df, group_col):
    rows = []
    for k, g in df.groupby(group_col, observed=True):
        rows.append(
            {
                'dimension': group_col,
                'bucket': str(k),
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead_base'),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead_base'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
            }
        )
    return pd.DataFrame(rows)

weighted_by_product = weighted_lgd(trade, 'product_type')
weighted_by_security = weighted_lgd(trade, 'security_level')
segment_summary = pd.concat([weighted_by_product, weighted_by_security], ignore_index=True)

base_vs_downturn = pd.DataFrame(
    [
        {
            'ead_weighted_lgd_base': exposure_weighted_average(trade, 'lgd_base_economic', 'ead_base'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(trade, 'lgd_downturn', 'ead_base'),
            'ead_weighted_lgd_final': exposure_weighted_average(trade, 'lgd_final_regulatory', 'ead_base'),
        }
    ]
)
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Weighted LGD by Product Type ===')
display(weighted_by_product.round(4))
print('=== Weighted LGD by Security Level ===')
display(weighted_by_security.round(4))
print('=== Base vs Downturn Comparison ===')
display(base_vs_downturn.round(4))


In [ ]:
# Sensitivity analysis: claim conversion and security support

def run_trade_sensitivity(df, claim_add=0.0, cash_sec_shift=0.0):
    x = df.copy()
    claim = (x['claim_probability_base'] + claim_add).clip(0.01, 1.0)
    cash_sec = (x['cash_security_pct'] + cash_sec_shift).clip(0.0, 1.0)

    ead_s = x['already_drawn_amount'] + x['contingent_commitment_amount'] * claim
    ead_s = np.maximum(ead_s, x['already_drawn_amount'])

    rec_months_s = (
        x['post_claim_recovery_months'] * (1.00 + 0.10 * claim_add - 0.08 * cash_sec_shift)
    ).clip(2.0, 48.0)

    lgd_base_s = (
        0.38 * x['realised_lgd']
        + 0.27 * (1.0 - cash_sec)
        + 0.14 * (1.0 - x['collateral_support_pct'])
        + 0.11 * (1.0 - x['customer_risk_strength'])
        + 0.04 * x['watchlist_flag']
        + 0.04 * ((rec_months_s - 6.0) / 18.0).clip(0.0, 1.0)
        + x['legal_processing_cost_pct']
    ).clip(0.05, 0.92)

    lgd_down_s = (
        lgd_base_s
        * (1.00 + 0.08 * claim + 0.10 * (1.0 - cash_sec) + 0.05 * x['watchlist_flag'])
    ).clip(0.0, 1.0)

    lgd_final_s = np.maximum(
        lgd_down_s + 0.015 + 0.010 * x['watchlist_flag'],
        x['supervisory_floor_proxy'],
    ).clip(0.0, 1.0)

    tmp = x.assign(ead_s=ead_s, lgd_base_s=lgd_base_s, lgd_down_s=lgd_down_s, lgd_final_s=lgd_final_s)
    return {
        'ead_weighted_lgd_base': exposure_weighted_average(tmp, 'lgd_base_s', 'ead_s'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(tmp, 'lgd_down_s', 'ead_s'),
        'ead_weighted_lgd_final': exposure_weighted_average(tmp, 'lgd_final_s', 'ead_s'),
    }

scenarios = [
    {'scenario': 'base', 'claim_add': 0.00, 'cash_sec_shift': 0.00},
    {'scenario': 'claim_prob_+10pp', 'claim_add': 0.10, 'cash_sec_shift': 0.00},
    {'scenario': 'cash_sec_-10pp', 'claim_add': 0.00, 'cash_sec_shift': -0.10},
    {'scenario': 'combined_stress', 'claim_add': 0.10, 'cash_sec_shift': -0.10},
]

sens_rows = []
for s in scenarios:
    m = run_trade_sensitivity(trade, claim_add=s['claim_add'], cash_sec_shift=s['cash_sec_shift'])
    sens_rows.append({'scenario': s['scenario'], **m})

sensitivity = pd.DataFrame(sens_rows)
print('=== Sensitivity Analysis ===')
display(sensitivity.round(4))


In [ ]:
# Monitoring, validation, charts, exports
trade['default_year'] = pd.to_datetime(trade['default_date']).dt.year
monitoring = (
    trade.groupby('default_year', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
                'mean_recovery_months': g['post_claim_recovery_months'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

validation_checks = pd.DataFrame(
    [
        {'check_name': 'contingent_commitment_non_negative', 'passed': (trade['contingent_commitment_amount'] >= 0).all()},
        {'check_name': 'already_drawn_non_negative', 'passed': (trade['already_drawn_amount'] >= 0).all()},
        {'check_name': 'drawn_not_above_limit', 'passed': (trade['already_drawn_amount'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'contingent_not_above_undrawn_headroom', 'passed': (trade['contingent_commitment_amount'] <= trade['undrawn_headroom_amount']).all()},
        {'check_name': 'claim_probability_bounds', 'passed': trade['claim_probability_base'].between(0, 1).all()},
        {'check_name': 'ead_base_not_below_drawn', 'passed': (trade['ead_base'] >= trade['already_drawn_amount']).all()},
        {'check_name': 'ead_downturn_not_below_drawn', 'passed': (trade['ead_downturn'] >= trade['already_drawn_amount']).all()},
        {'check_name': 'ead_base_not_above_limit', 'passed': (trade['ead_base'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'ead_downturn_not_above_limit', 'passed': (trade['ead_downturn'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'recovery_months_positive', 'passed': (trade['post_claim_recovery_months'] > 0).all()},
        {'check_name': 'downturn_recovery_not_shorter', 'passed': (trade['post_claim_recovery_months_downturn'] >= trade['post_claim_recovery_months']).all()},
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weighted_by_product['bucket'], weighted_by_product['ead_weighted_lgd_final'], color='#c44e52')
ax.set_title('Trade / Contingent LGD: Weighted Final LGD by Product Type')
ax.set_xlabel('Product Type')
ax.set_ylabel('EAD-weighted Final LGD')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'trade_contingent_weighted_lgd_by_product.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ead_conversion_summary['security_level'], ead_conversion_summary['ead_weighted_conversion_uplift_pct'], color='#55a868')
ax.set_title('Trade / Contingent EAD Conversion Uplift by Security Level')
ax.set_xlabel('Security Level')
ax.set_ylabel('EAD Conversion Uplift %')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'trade_contingent_ead_conversion_uplift.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

loan_level_output = trade[
    [
        'loan_id', 'facility_type', 'security_type', 'product_type', 'transaction_type_proxy',
        'is_contingent_flag', 'tenor_months', 'industry', 'industry_risk_bucket',
        'facility_limit_proxy', 'undrawn_headroom_amount',
        'contingent_commitment_amount', 'already_drawn_amount', 'claim_probability_base', 'claim_probability_downturn',
        'cash_security_pct', 'collateral_support_pct', 'security_level', 'customer_risk_strength',
        'ead_base', 'ead_downturn', 'ead_conversion_uplift_pct',
        'post_claim_recovery_months', 'post_claim_recovery_months_downturn',
        'legal_processing_cost_pct', 'realised_lgd', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory',
    ]
].copy()

loan_level_output.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_segment_summary.csv'), index=False)
ead_conversion_summary.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_ead_conversion_summary.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_base_vs_downturn.csv'), index=False)
sensitivity.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_sensitivity.csv'), index=False)
monitoring.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_monitoring_by_year.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_validation_checks.csv'), index=False)

print('=== Validation Checks ===')
display(validation_checks)

print('Saved trade/contingent outputs:')
print('- trade_contingent_loan_level_output.csv')
print('- trade_contingent_segment_summary.csv')
print('- trade_contingent_ead_conversion_summary.csv')
print('- trade_contingent_base_vs_downturn.csv')
print('- trade_contingent_sensitivity.csv')
print('- trade_contingent_monitoring_by_year.csv')
print('- trade_contingent_validation_checks.csv')
